# 📚 Hello Clova Agent — Wiki + 도메인 덱 생성

**나의 도메인 문서 → wiki 구축 → 도메인 지식 기반 발표 자동 생성**

---

## 이 노트북의 흐름

```
[내 문서 업로드]  →  [wiki 구축]  →  [도메인 덱 생성]
 PDF / ipynb          convert          Gradio 앱
                       ingest
```

| 셀 | 역할 |
|----|------|
| **셀 1/7** | Google Drive 마운트 — LLM 모델 캐시 저장소 |
| **셀 2/7** | Ollama 설치 + 모델 준비 (Drive 캐시 사용 시 빠름) |
| **셀 3/7** | 프로젝트 코드 준비 (git clone / pull) |
| **셀 4/7** | 의존성 설치 + 환경변수 설정 |
| **셀 5/7** | 도메인 문서 업로드 + 변환 (convert) |
| **셀 6/7** | wiki 구축 (ingest) |
| **셀 7/7** | 도메인 덱 생성 에이전트 실행 |

---

## 💡 LLM 다운로드는 최초 1회만

> **왜 시간이 걸리나?**  
> LLM 모델 파일(가중치)은 수 GB 크기입니다 — Gradle의 JAR 의존성처럼  
> 최초 1회 다운로드가 필요합니다.  
>  
> **이 노트북의 해결책: Google Drive 캐시**  
> 모델을 Drive에 저장해두면, 다음 Colab 세션부터는  
> Drive에서 복사만 하면 되어 **약 1-3분**으로 단축됩니다.

| 구분 | 최초 실행 | 이후 실행 |
|------|-----------|----------|
| Drive 캐시 없음 | ~10분 (다운로드) | ~10분 (매번 다운로드) |
| **Drive 캐시 있음** | ~10분 (다운로드 후 저장) | **~2분 (복사만)** |

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 1/7  ▌ Google Drive 마운트 (LLM 모델 캐시)
#
# [개념] 왜 Drive를 마운트하나?
#   Colab VM은 세션이 끊기면 초기화됩니다.
#   Drive는 영구 저장소이므로 LLM 모델을 여기 저장해두면
#   다음 세션에서 재다운로드 없이 바로 사용할 수 있습니다.
#
#   모델 캐시 위치: Google Drive/hello-clova-agent-cache/
#
# [선택사항]
#   Drive 없이도 실행 가능 — 단 매 세션마다 모델 재다운로드 필요
#   USE_DRIVE = False 로 설정하면 Drive 없이 진행합니다.
# ══════════════════════════════════════════════════════════════════════

import os

USE_DRIVE = True   # ← Drive 사용 여부 (False로 바꾸면 캐시 없이 진행)

DRIVE_CACHE = "/content/drive/MyDrive/hello-clova-agent-cache/ollama"
OLLAMA_MODELS = os.path.expanduser("~/.ollama/models")

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    print(f"✅ Drive 마운트 완료")
    print(f"   모델 캐시 경로: {DRIVE_CACHE}")
else:
    print("⚠️ Drive 미사용 — 매 세션마다 모델 재다운로드 필요")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 2/7  ▌ Ollama 설치 + 모델 준비
#
# Drive 캐시가 있으면: Drive → 로컬 복사 (~1-3분)
# Drive 캐시 없으면:  Ollama pull (~5-10분) → Drive 저장
#
# [개념] API 서버 (Ollama)
#   LLM을 OpenAI 호환 HTTP API로 제공합니다.
#   에이전트는 http://localhost:11434/v1/chat/completions 로 호출합니다.
# ══════════════════════════════════════════════════════════════════════

import subprocess, time, urllib.request, shutil, os

MODEL_NAME = "qwen2.5:7b"   # ← 변경 가능 (예: "qwen2.5:3b" for 가벼운 버전)
os.environ["LLM_MODEL"] = MODEL_NAME

# 1) Ollama 설치
print("📦 Ollama 설치 중...")
!sudo apt-get update -qq && sudo apt-get install -y zstd curl -qq
!curl -fsSL https://ollama.com/install.sh | sh

# 2) 기존 프로세스 정리 후 서버 시작
subprocess.run(["pkill", "-9", "ollama"], capture_output=True)
time.sleep(1)
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

print("⏳ Ollama 서버 시작 대기 중", end="")
for _ in range(30):
    try:
        urllib.request.urlopen("http://localhost:11434/api/version", timeout=1)
        print(" ✅"); break
    except: print(".", end="", flush=True); time.sleep(1)

# 3) 모델 준비 (Drive 캐시 우선)
DRIVE_CACHE = "/content/drive/MyDrive/hello-clova-agent-cache/ollama"
OLLAMA_MODELS = os.path.expanduser("~/.ollama/models")

drive_model_dir = os.path.join(DRIVE_CACHE, "models")
if os.path.exists(drive_model_dir) and os.listdir(drive_model_dir):
    print(f"\n📂 Drive 캐시에서 모델 복사 중... (~1-3분)")
    os.makedirs(OLLAMA_MODELS, exist_ok=True)
    !cp -rn {drive_model_dir}/. {OLLAMA_MODELS}/
    print(f"✅ 모델 복사 완료 (Drive → 로컬)")
else:
    print(f"\n📥 모델 다운로드 중: {MODEL_NAME} (~5-10분, 최초 1회)")
    !ollama pull {MODEL_NAME}
    if os.environ.get("USE_DRIVE", "True") == "True" and os.path.exists("/content/drive"):
        print("💾 Drive에 모델 캐시 저장 중...")
        os.makedirs(drive_model_dir, exist_ok=True)
        !cp -rn {OLLAMA_MODELS}/. {drive_model_dir}/
        print("✅ Drive 캐시 저장 완료 — 다음 세션부터 빠르게 로드됩니다")

print(f"\n✅ LLM 준비 완료: {MODEL_NAME}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 3/7  ▌ 프로젝트 코드 준비 (git clone / pull)
# ══════════════════════════════════════════════════════════════════════

import os

REPO_URL = "https://github.com/machine-artisan/Hello-Clova-Agent.git"
REPO_DIR = "Hello-Clova-Agent"

if os.path.exists(REPO_DIR):
    print("🔄 최신 코드로 업데이트 중...")
    !git -C {REPO_DIR} pull
else:
    print("📂 프로젝트 다운로드 중...")
    !git clone {REPO_URL}

os.chdir(REPO_DIR)
print(f"\n✅ 현재 경로: {os.getcwd()}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 4/7  ▌ 의존성 설치 + 환경변수 설정
# ══════════════════════════════════════════════════════════════════════

import os

print("📦 Python 패키지 설치 중...")
!pip install -r requirements.txt -q
!pip install pymupdf nbformat -q   # wiki_agent용 추가 의존성

os.environ["LLM_API_BASE"] = "http://localhost:11434/v1"
os.environ["LLM_API_KEY"]  = "EMPTY"
# LLM_MODEL은 셀 2에서 이미 설정됨

print("\n✅ 설치 및 환경변수 설정 완료")
print(f"   LLM_API_BASE = {os.environ['LLM_API_BASE']}")
print(f"   LLM_MODEL    = {os.environ.get('LLM_MODEL', 'qwen2.5:7b')}")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 5/7  ▌ 도메인 문서 업로드 + 변환 (convert)
#
# [방법 A] Colab 파일 업로더 (소용량 문서)
# [방법 B] Google Drive에서 복사 (대용량 / 다수 파일)
#
# 지원 형식: PDF (.pdf), Jupyter Notebook (.ipynb)
# 변환 결과: sources/md/ 에 저장됨
# ══════════════════════════════════════════════════════════════════════

import os, shutil
from pathlib import Path

# ── 방법 선택 ─────────────────────────────────────────────────────────
UPLOAD_METHOD = "uploader"   # "uploader" 또는 "drive"
# Drive 사용 시: 아래 경로에 파일을 넣어두세요
DRIVE_DOCS_DIR = "/content/drive/MyDrive/hello-clova-agent-docs/"
# ─────────────────────────────────────────────────────────────────────

RAW_DIR = Path("sources/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

if UPLOAD_METHOD == "uploader":
    from google.colab import files
    print("📎 업로드할 파일을 선택하세요 (PDF 또는 .ipynb):")
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = RAW_DIR / name
        dest.write_bytes(data)
        print(f"  → sources/raw/{name}")

elif UPLOAD_METHOD == "drive":
    src_files = list(Path(DRIVE_DOCS_DIR).glob("*.pdf")) + \
                list(Path(DRIVE_DOCS_DIR).glob("*.ipynb"))
    for f in src_files:
        shutil.copy(str(f), str(RAW_DIR / f.name))
        print(f"  → sources/raw/{f.name}")

# 변환 실행
import sys
sys.path.insert(0, ".")
from wiki_agent.convert import convert

raw_files = [f for f in RAW_DIR.iterdir() if f.suffix in (".pdf", ".ipynb")]
if raw_files:
    print(f"\n🔄 {len(raw_files)}개 파일 변환 중...")
    for f in raw_files:
        convert(str(f))
    print("\n✅ 변환 완료 → sources/md/ 확인")
    !ls sources/md/
else:
    print("⚠️ sources/raw/ 에 변환할 파일이 없습니다.")

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 6/7  ▌ wiki 구축 (ingest)
#
# sources/md/ 의 Markdown 파일을 LLM으로 분석하여
# wiki/domain.md, wiki/stack.md, wiki/glossary.md 를 업데이트합니다.
#
# 완료 후 처리된 파일은 sources/processed/ 로 이동됩니다.
# ══════════════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, ".")

from wiki_agent.ingest import ingest_all

print("📖 wiki 구축 시작...")
ingest_all()

# wiki 구축 결과 확인
print("\n📋 wiki 현황:")
!wc -l wiki/*.md

## 🏗️ 시스템 아키텍처

```
┌──────────────────────────────────────────────────────────┐
│                     브라우저 (여러분의 PC)                │
└──────────────────────┬───────────────────────────────────┘
                       │  HTTP (공개 URL)
┌──────────────────────▼───────────────────────────────────┐
│        🌐 웹서버  —  Gradio  (포트 7860)                 │
└──────────────────────┬───────────────────────────────────┘
                       │  Python 함수 호출
┌──────────────────────▼───────────────────────────────────┐
│        ⚙️  앱서버  —  LangGraph 파이프라인               │
│   [wiki 컨텍스트 자동 주입] ← wiki_agent/wiki_loader.py  │
└──────────────────────┬───────────────────────────────────┘
                       │  HTTP (OpenAI API)
┌──────────────────────▼───────────────────────────────────┐
│        🤖 API서버  —  Ollama  (포트 11434)               │
└──────────────────────┬───────────────────────────────────┘
                       │  GPU
┌──────────────────────▼───────────────────────────────────┐
│        🧠 LLM  —  Qwen 2.5 7B  (Colab GPU)              │
└──────────────────────────────────────────────────────────┘
```

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# 셀 7/7  ▌ 도메인 덱 생성 에이전트 실행 🚀
#
# wiki에 구축된 도메인 지식이 슬라이드 생성 시 자동으로 주입됩니다.
# Gradio UI의 프롬프트 창에 발표 주제를 입력하면 됩니다.
#
# 공개 URL 예) Running on public URL: https://xxxx.gradio.live
# ══════════════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, ".")
from wiki_agent.wiki_loader import has_wiki, load_wiki

if has_wiki():
    print("✅ Wiki 컨텍스트 로드됨 — 도메인 지식이 슬라이드 생성에 주입됩니다")
    wiki_preview = load_wiki()[:300]
    print(f"   미리보기: {wiki_preview[:200]}...")
else:
    print("⚠️ Wiki가 비어 있습니다 — 범용 모드로 실행")
    print("   (셀 5, 6을 먼저 실행하면 도메인 지식이 주입됩니다)")

print("\n🚀 에이전트 시작 중...")
print("   아래에 공개 URL이 출력되면 브라우저에서 접속하세요 👇")
print()

!python ui/app.py

---

## 📂 프로젝트 구조

```
Hello-Clova-Agent/
│
├── agent/           ← 덱 생성 파이프라인 (LangGraph 4-노드)
├── wiki_agent/      ← wiki 관리 파이프라인
│   ├── convert.py   ← PDF/ipynb → MD
│   ├── ingest.py    ← MD → wiki/*.md
│   └── wiki_loader.py ← wiki → 덱 생성 컨텍스트 주입
│
├── wiki/            ← 사용자 도메인 지식 베이스 (누적)
│   ├── domain.md    ← 핵심 개념
│   ├── stack.md     ← 기술/도구
│   └── glossary.md  ← 용어집
│
├── sources/
│   ├── raw/         ← 원본 문서 투입 (PDF, ipynb)
│   ├── md/          ← 변환된 MD (ingest 대기)
│   └── processed/   ← ingest 완료 아카이브
│
└── ui/app.py        ← Gradio 웹 UI
```

---
*Hello Clova Agent · Phase 2 Wiki · LangGraph + Ollama + wiki_agent + Reveal.js*